# 03 — Temporal Analysis & Leakage

**Owner:** Bandara's lane — timestamp/cycle analysis, event-window leakage study, split strategy.

This is the notebook Eval 1 is won or lost on (see `CLAUDE.md`). Output of this notebook: the final eligible-feature list and the split decision, handed off to notebook 04 (`preprocessing`).

Remember: every important choice gets a decision-log cell below it (what we decided, evidence, alternative considered, why rejected).

In [ ]:
import sys
from pathlib import Path

# Jupyter's kernel CWD is notebooks/, so add the repo root to sys.path
# before importing anything from src/.
sys.path.append(str(Path.cwd().parent))

In [ ]:
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import RANDOM_STATE, FIGURES_DIR, PROCESSED_DATA_DIR
from src.pipeline import load_data

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

In [ ]:
df = load_data()
df = df.sort_values("Timestamp").reset_index(drop=True)
df.shape

## Cycle stats

TODO: rows per cycle (plan expects 11-102, median ~25), reading cadence (~1/sec), and locate the ~5 hour gap between 08:17 and 15:36 on 26 Oct 2022.

In [ ]:
df.groupby("cycle").size().describe()

In [ ]:
# TODO: confirm the reading cadence and locate the large gap
df["Timestamp"].diff().describe()

## Stop episodes

TODO: group consecutive `Robot_ProtectiveStop == 1` rows into episodes. Plan expects 278 stop rows -> 108 episodes (median 2 rows, max 12), occurring in 78 of 240 cycles. Confirm and adjust if the raw numbers differ.

In [ ]:
is_stop = df["Robot_ProtectiveStop"] == 1
episode_id = (is_stop != is_stop.shift()).cumsum()
episodes = df[is_stop].groupby(episode_id[is_stop]).size()
print("stop rows:", is_stop.sum())
print("episodes:", episodes.shape[0])
episodes.describe()

## Timeline of stops

TODO: plot stop occurrences over the day's timeline. Save to `FIGURES_DIR / "fig03_stop_timeline.png"`.

In [ ]:
# TODO: timeline plot

## Event-window leakage study (±10 rows around each stop)

TODO: for each stop episode, look at feature values from -10 to +10 rows around it. The plan's headline finding: median total joint speed is ~0.002 at stop rows vs ~0.056 at normal rows — same-row speed readings mostly describe a robot that has *already* stopped, not one about to stop. Confirm this and identify which other features carry the same same-row leakage.

In [ ]:
speed_cols = [c for c in df.columns if c.startswith("Speed_J")]
# TODO: decide the aggregation (e.g. sum of abs, or L2 norm) and justify it in the decision log
df["total_joint_speed"] = df[speed_cols].abs().sum(axis=1)
df.groupby("Robot_ProtectiveStop")["total_joint_speed"].median()

In [ ]:
# TODO: build ±10-row windows around each stop episode and plot feature trajectories

## grip_lost cross-tab

TODO: overlap between `grip_lost` and `Robot_ProtectiveStop`. Plan expects only 3 overlapping rows out of 243 grip-loss rows — confirm it is a weak predictor / separate event, not a leakage source.

In [ ]:
pd.crosstab(df["grip_lost"], df["Robot_ProtectiveStop"])

## Final eligible-feature list

TODO: decide which raw/engineered features are safe to use (exclude same-row leakage features identified above, or redefine them using only prior-row/lagged values). This list is the handoff to notebook 04.

In [ ]:
# TODO: fill in the final list once the leakage study above is done
ELIGIBLE_FEATURES = []

with open(PROCESSED_DATA_DIR / "eligible_features.json", "w") as f:
    json.dump(ELIGIBLE_FEATURES, f, indent=2)

ELIGIBLE_FEATURES

## Split strategy decision

TODO: decide and justify StratifiedGroupKFold-by-cycle vs a chronological split (open decision in `CLAUDE.md`). Save the split (train/test row indices or cycle assignments) to `data/processed/` so notebook 04 and every model notebook reuse the exact same split.

In [ ]:
# TODO: split implementation, then persist e.g. to data/processed/train_test_split.json

## Decision log

### Decision: <what we decided>
- **Evidence:** <figure/number>
- **Alternative considered:** <...>
- **Why rejected:** <...>